# CV quality inspection: transfer-learning models (Colab, T4 GPU)

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. About 15–20 minutes.
At the end a file `cv_outputs.zip` downloads. Unzip it in the repo root: it fills `models/` and
`reports/predictions/`.

* **B · linear probe:** frozen ImageNet EfficientNet-B0, logistic regression on its 1,280 pooled features.
* **C · fine-tuned:** EfficientNet-B0, new 1-logit head; head first, then the last blocks unfrozen; train-only augmentation.

Model choice and early stopping use **validation only**. Test predictions are written once, at the end.

In [ ]:
!pip -q install kagglehub onnx onnxruntime
import base64, gzip, hashlib, io, json, os, random, time, zipfile
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torchvision
from PIL import Image
from torchvision import transforms as T
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(DEV, torch.__version__, torchvision.__version__)
OUT = Path("cv_outputs"); (OUT / "models").mkdir(parents=True, exist_ok=True); (OUT / "reports/predictions").mkdir(parents=True, exist_ok=True)

## 1 · The split manifest (embedded) and its checksum

In [ ]:
MANIFEST_SHA = "8c5f47328694a174619c9b6a454b9ab22c8a37be93a7ef3cfa94164b9c9e9bbd"
raw = gzip.decompress(base64.b64decode("..."))
assert hashlib.sha256(raw).hexdigest() == MANIFEST_SHA, "manifest changed"
m = pd.read_csv(io.BytesIO(raw))
print(m.groupby(["split", "label"]).size().unstack())

## 2 · Dataset from Kaggle, every file checked against the manifest

In [ ]:
import kagglehub
# Public dataset: normally downloads without a login. If it asks for credentials, run
# kagglehub.login() in a new cell (username + API key from kaggle.com > Settings > API) and re-run this cell.
root = Path(kagglehub.dataset_download("ravirajsinh45/real-life-industrial-dataset-of-casting-product"))
def sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
m["file"] = [root / p for p in m["path"]]
missing = [str(p) for p in m["file"] if not p.exists()]
assert not missing, missing[:5]
bad = (m["file"].map(sha256) != m["file_sha256"]).sum()
assert bad == 0, f"{bad} files differ from the local copy"
print("all", len(m), "files match")

## 3 · Preprocessing (identical to `src/preprocess.py`) and train-only augmentation

In [ ]:
"""The single preprocessing function used everywhere a model sees an image.

The Colab notebook copies this exact code (make_colab_notebook.py inlines this file), so
training, evaluation, the API and the app all feed the network identical tensors.
Augmentation is NOT here: it is applied only to training images, inside the notebook.
"""
import numpy as np
from PIL import Image

IMG_SIZE = 224
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def to_gray_image(img) -> Image.Image:
    """Path / PIL image / array -> grayscale PIL image (the camera is monochrome)."""
    if isinstance(img, (str, bytes)) or hasattr(img, "__fspath__"):
        img = Image.open(img)
    elif isinstance(img, np.ndarray):
        img = Image.fromarray(img)
    return img.convert("L")


def to_input(img) -> np.ndarray:
    """-> float32 (3, 224, 224): resize (bilinear), scale to [0, 1], repeat to 3 channels, ImageNet-normalise."""
    g = to_gray_image(img).resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    x = np.asarray(g, dtype=np.float32) / 255.0
    x = np.repeat(x[None], 3, axis=0)
    return ((x - MEAN[:, None, None]) / STD[:, None, None]).astype(np.float32)


class Rot90:
    # the part is round and shot from the top: any 90-degree turn is a physically real view
    def __call__(self, img): return img.rotate(90 * random.randint(0, 3))

AUG = T.Compose([Rot90(), T.RandomHorizontalFlip(), T.RandomRotation(15, fill=128),
                 T.ColorJitter(brightness=0.2, contrast=0.2), T.RandomApply([T.GaussianBlur(3)], p=0.2)])

class DS(torch.utils.data.Dataset):
    def __init__(self, df, train): self.df, self.train = df.reset_index(drop=True), train
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        img = to_gray_image(self.df.file[i]).resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
        if self.train: img = AUG(img)          # augmentation only ever touches training images
        return torch.from_numpy(to_input(img)), torch.tensor(float(self.df.label[i]))

split = {s: m[m.split == s].reset_index(drop=True) for s in ["train", "val", "test"]}
def loader(s, train=False, bs=32):
    g = torch.Generator().manual_seed(SEED)
    return torch.utils.data.DataLoader(DS(split[s], train), batch_size=bs, shuffle=train, num_workers=2, generator=g)

## 4 · B · linear probe on frozen ImageNet features

In [ ]:
W = torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1
def backbone():
    net = torchvision.models.efficientnet_b0(weights=W)
    return net

@torch.no_grad()
def pooled(net, s):
    net.eval(); feats = []
    for x, _ in loader(s):
        f = net.features(x.to(DEV)); feats.append(net.avgpool(f).flatten(1).cpu().numpy())
    return np.concatenate(feats)

net = backbone().to(DEV)
E = {s: pooled(net, s) for s in ["train", "val", "test"]}
y = {s: split[s].label.values for s in split}
probe_log = []
for C in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]:
    clf = LogisticRegression(C=C, max_iter=5000).fit(E["train"], y["train"])
    probe_log.append({"C": C, "val_ap": average_precision_score(y["val"], clf.predict_proba(E["val"])[:, 1])})
probe_log = pd.DataFrame(probe_log); print(probe_log)
best_C = float(probe_log.loc[probe_log.val_ap.idxmax(), "C"])
clf = LogisticRegression(C=best_C, max_iter=5000).fit(E["train"], y["train"])
pd.concat([split[s][["path", "label", "split"]].assign(score=clf.predict_proba(E[s])[:, 1]) for s in ["val", "test"]]
          ).to_csv(OUT / "reports/predictions/linear_probe.csv", index=False)
probe_log.to_csv(OUT / "reports/tuning_linear_probe.csv", index=False)

## 5 · C · fine-tune EfficientNet-B0 (early stopping on validation average precision)

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        b = backbone()
        self.features, self.pool = b.features, nn.AdaptiveAvgPool2d(1)
        self.drop, self.head = nn.Dropout(0.3), nn.Linear(1280, 1)
    def forward(self, x):
        return self.head(self.drop(self.pool(self.features(x)).flatten(1))).squeeze(1)

@torch.no_grad()
def predict(model, s):
    model.eval(); out = []
    for x, _ in loader(s): out.append(model(x.to(DEV)).float().cpu().numpy())
    return np.concatenate(out)

def run_phase(model, params, epochs, log, phase, patience=4):
    opt = torch.optim.AdamW(params, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    lossf = nn.BCEWithLogitsLoss()
    best, best_state, wait = -1, None, 0
    for ep in range(epochs):
        model.train(); t0, tot = time.time(), 0.0
        for x, yb in loader("train", train=True):
            opt.zero_grad(); loss = lossf(model(x.to(DEV)), yb.to(DEV)); loss.backward(); opt.step(); tot += loss.item() * len(x)
        sched.step()
        z = predict(model, "val"); yv = y["val"]
        vloss = float(lossf(torch.tensor(z), torch.tensor(yv, dtype=torch.float32)))
        ap = average_precision_score(yv, z)
        log.append({"phase": phase, "epoch": ep + 1, "train_loss": tot / len(split["train"]), "val_loss": vloss,
                    "val_ap": ap, "val_auc": roc_auc_score(yv, z), "sec": time.time() - t0})
        print(log[-1])
        if ap > best + 1e-4:
            best, best_state, wait = ap, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    return best

model, log = Net().to(DEV), []
for p in model.features.parameters(): p.requires_grad = False          # phase 1: head only
run_phase(model, [{"params": model.head.parameters(), "lr": 1e-3}], 4, log, "head")
for p in model.features[5:].parameters(): p.requires_grad = True       # phase 2: last blocks too
run_phase(model, [{"params": model.features[5:].parameters(), "lr": 1e-4},
                  {"params": model.head.parameters(), "lr": 5e-4}], 15, log, "fine-tune")
pd.DataFrame(log).to_csv(OUT / "reports/train_log.csv", index=False)
best_row = max([r for r in log if r["phase"] == "fine-tune"], key=lambda r: r["val_ap"])
print("best fine-tune epoch:", best_row)

## 6 · Predictions (test written once, after every choice above is fixed)

In [ ]:
preds = []
for s in ["val", "test"]:
    z = predict(model, s)
    preds.append(split[s][["path", "label", "split"]].assign(logit=z, score=1 / (1 + np.exp(-z))))
pd.concat(preds).to_csv(OUT / "reports/predictions/finetuned.csv", index=False)
print(pd.concat(preds).groupby("split").apply(lambda d: average_precision_score(d.label, d.score)))

## 7 · Export to ONNX for CPU serving

Two outputs: the defect **logit** and the last **feature map** (1280 × 7 × 7). Because the head is
global-average-pool + one linear layer, the class activation map is exactly
`sum_c w_c * feature_map_c`, computed locally with the saved head weights; no gradients needed.

In [ ]:
class Export(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):
        f = self.m.features(x)
        return self.m.head(self.m.pool(f).flatten(1)).squeeze(1), f

model.eval().cpu()
dummy = torch.from_numpy(np.stack([to_input(split["val"].file[i]) for i in range(4)]))
torch.onnx.export(Export(model), dummy, OUT / "models/model.onnx", input_names=["image"], output_names=["logit", "features"],
                  dynamic_axes={"image": {0: "n"}, "logit": {0: "n"}, "features": {0: "n"}}, opset_version=17, dynamo=False)
np.savez(OUT / "models/head.npz", w=model.head.weight.detach().numpy()[0], b=model.head.bias.detach().numpy())

import onnxruntime as ort
sess = ort.InferenceSession(str(OUT / "models/model.onnx"))
z_onnx = sess.run(["logit"], {"image": dummy.numpy()})[0]
with torch.no_grad(): z_torch = model(dummy).numpy()
diff = float(np.abs(z_onnx - z_torch).max()); print("max |onnx - torch| logit:", diff); assert diff < 1e-3

meta = {"backbone": "efficientnet_b0 IMAGENET1K_V1", "torch": torch.__version__, "torchvision": torchvision.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "manifest_sha256": MANIFEST_SHA,
        "linear_probe_C": best_C, "best_finetune_epoch": best_row["epoch"], "best_val_ap": best_row["val_ap"],
        "onnx_torch_max_diff": diff, "epochs_run": len(log), "train_minutes": round(sum(r["sec"] for r in log) / 60, 1),
        "onnx_mb": round(os.path.getsize(OUT / "models/model.onnx") / 1e6, 1)}
(OUT / "models/colab_meta.json").write_text(json.dumps(meta, indent=2)); print(meta)

## 8 · Download

In [ ]:
import shutil
shutil.make_archive("cv_outputs", "zip", OUT)
from google.colab import files
files.download("cv_outputs.zip")